# AMI Subset ASR Pipeline (Whisper)

This notebook runs a resumable Whisper ASR pipeline over the AMI subset audio. It writes per-file JSON outputs, per-file transcripts, and a checkpoint index so you can resume without reprocessing completed files.

Notes:
- No randomness is used, and files are processed in deterministic order.
- Edit the configuration cell to change paths, model size, and device settings.

In [1]:
from __future__ import annotations

import hashlib
import json
import logging
import os
import time
from datetime import datetime
from pathlib import Path
from typing import Iterable, Optional

import pandas as pd
from faster_whisper import WhisperModel

# ---- Paths (edit here if your folder structure differs) ----
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_AUDIO_DIR = PROJECT_ROOT / 'data' / 'raw_audio'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'asr_outputs'

RAW_JSON_DIR = OUTPUT_ROOT / 'raw'
SEGMENTS_DIR = OUTPUT_ROOT / 'segments'
TRANSCRIPTS_DIR = OUTPUT_ROOT / 'transcripts'
LOGS_DIR = OUTPUT_ROOT / 'logs'
CHECKPOINTS_DIR = OUTPUT_ROOT / 'checkpoints'

for directory in [RAW_JSON_DIR, SEGMENTS_DIR, TRANSCRIPTS_DIR, LOGS_DIR, CHECKPOINTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ---- Configuration ----
AUDIO_EXTS = {'.wav'}
SESSION_PATTERN = r'^(?P<session>[A-Z]{2}\d{4}[a-d])'

MODEL_NAME = 'medium'
DEVICE = 'cpu'  # Use 'cuda' if you have a supported GPU
COMPUTE_TYPE = 'int8'  # Try 'float16' for CUDA, or 'int8' for CPU

LANGUAGE = None  # Set to a language code like 'en' if you want to force language
TASK = 'transcribe'
BEAM_SIZE = 5
TEMPERATURE = 0.0
VAD_FILTER = False
WORD_TIMESTAMPS = False

SKIP_IF_CHECKPOINTED = True
RETRY_FAILED = False

CHECKPOINT_INDEX = CHECKPOINTS_DIR / 'asr_index.csv'
RUN_ID = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
LOG_PATH = LOGS_DIR / f'asr_run_{RUN_ID}.log'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(LOG_PATH),
        logging.StreamHandler(),
    ],
)

os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

/Users/adityagoyal/SRH/Thesis Research/thesis_code/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/cg/1qj3kl_12w168jf9m109fgrr0000gn/T/ipykernel_49195/2270402826.py:48: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  RUN_ID = datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')


'false'

In [2]:
def iter_files(root: Path, exts: Optional[set[str]] = None) -> Iterable[Path]:
    if not root.exists():
        return []
    for path in sorted(root.rglob('*')):
        if path.is_file() and (exts is None or path.suffix.lower() in exts):
            yield path


def parse_session_id_from_name(name: str) -> Optional[str]:
    match = re.match(SESSION_PATTERN, name)
    return match.group('session') if match else None


def make_file_id(relative_path: str) -> str:
    return hashlib.sha1(relative_path.encode('utf-8')).hexdigest()[:12]


def load_checkpoint(path: Path) -> pd.DataFrame:
    columns = [
        'file_id',
        'relative_path',
        'session_id',
        'status',
        'model_name',
        'device',
        'compute_type',
        'language',
        'task',
        'runtime_seconds',
        'audio_duration_seconds',
        'raw_json_path',
        'segments_json_path',
        'transcript_path',
        'error',
        'updated_at',
    ]
    if not path.exists():
        return pd.DataFrame(columns=columns)
    df = pd.read_csv(path)
    if df.empty:
        return pd.DataFrame(columns=columns)
    if 'updated_at' in df.columns:
        df = df.sort_values('updated_at').drop_duplicates('file_id', keep='last')
    return df


def save_checkpoint(df: pd.DataFrame, path: Path) -> None:
    df.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    with path.open('w', encoding='utf-8') as handle:
        json.dump(payload, handle, ensure_ascii=True, indent=2)


def write_text(path: Path, text: str) -> None:
    with path.open('w', encoding='utf-8') as handle:
        handle.write(text)


def build_segments_payload(segments) -> list[dict]:
    payload = []
    for segment in segments:
        payload.append({
            'start': float(segment.start),
            'end': float(segment.end),
            'text': segment.text.strip(),
            'avg_logprob': float(segment.avg_logprob),
            'compression_ratio': float(segment.compression_ratio),
            'no_speech_prob': float(segment.no_speech_prob),
        })
    return payload

In [3]:
import re

logging.info('Loading checkpoint index: %s', CHECKPOINT_INDEX)
checkpoint_df = load_checkpoint(CHECKPOINT_INDEX)
checkpoint_df = checkpoint_df.set_index('file_id', drop=False)

audio_files = list(iter_files(RAW_AUDIO_DIR, exts=AUDIO_EXTS))
logging.info('Discovered %s audio files', len(audio_files))

if not audio_files:
    raise FileNotFoundError(f'No audio files found under {RAW_AUDIO_DIR}')

logging.info('Initializing Whisper model: %s', MODEL_NAME)
model = WhisperModel(MODEL_NAME, device=DEVICE, compute_type=COMPUTE_TYPE)

success_count = 0
skip_count = 0
fail_count = 0

for index, audio_path in enumerate(audio_files, start=1):
    relative_path = audio_path.relative_to(PROJECT_ROOT).as_posix()
    session_id = parse_session_id_from_name(audio_path.name)
    file_id = make_file_id(relative_path)

    if file_id in checkpoint_df.index and SKIP_IF_CHECKPOINTED:
        existing_status = str(checkpoint_df.loc[file_id].get('status', '')).lower()
        if existing_status != 'failed' or not RETRY_FAILED:
            logging.info('[%s/%s] Skipping (checkpointed): %s', index, len(audio_files), relative_path)
            skip_count += 1
            continue

    logging.info('[%s/%s] Transcribing: %s', index, len(audio_files), relative_path)
    start_time = time.perf_counter()

    raw_json_path = RAW_JSON_DIR / f'{file_id}.json'
    segments_json_path = SEGMENTS_DIR / f'{file_id}.json'
    transcript_path = TRANSCRIPTS_DIR / f'{file_id}.txt'

    try:
        segments, info = model.transcribe(
            str(audio_path),
            language=LANGUAGE,
            task=TASK,
            beam_size=BEAM_SIZE,
            temperature=TEMPERATURE,
            vad_filter=VAD_FILTER,
            word_timestamps=WORD_TIMESTAMPS,
        )

        segments_payload = build_segments_payload(segments)
        full_text = ' '.join([segment['text'] for segment in segments_payload]).strip()

        runtime_seconds = time.perf_counter() - start_time

        raw_payload = {
            'file_id': file_id,
            'session_id': session_id,
            'relative_path': relative_path,
            'audio_path': str(audio_path),
            'model_name': MODEL_NAME,
            'device': DEVICE,
            'compute_type': COMPUTE_TYPE,
            'language': info.language if info else None,
            'task': TASK,
            'audio_duration_seconds': float(info.duration) if info else None,
            'runtime_seconds': float(runtime_seconds),
            'text': full_text,
            'segments': segments_payload,
            'created_at': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
            'status': 'success',
        }

        write_json(raw_json_path, raw_payload)
        write_json(segments_json_path, {
            'file_id': file_id,
            'session_id': session_id,
            'relative_path': relative_path,
            'segments': segments_payload,
        })
        write_text(transcript_path, full_text)

        checkpoint_row = {
            'file_id': file_id,
            'relative_path': relative_path,
            'session_id': session_id,
            'status': 'success',
            'model_name': MODEL_NAME,
            'device': DEVICE,
            'compute_type': COMPUTE_TYPE,
            'language': info.language if info else None,
            'task': TASK,
            'runtime_seconds': float(runtime_seconds),
            'audio_duration_seconds': float(info.duration) if info else None,
            'raw_json_path': str(raw_json_path),
            'segments_json_path': str(segments_json_path),
            'transcript_path': str(transcript_path),
            'error': None,
            'updated_at': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
        }
        checkpoint_df.loc[file_id] = checkpoint_row
        save_checkpoint(checkpoint_df.reset_index(drop=True), CHECKPOINT_INDEX)

        success_count += 1
    except Exception as exc:
        runtime_seconds = time.perf_counter() - start_time
        logging.exception('Failed to transcribe %s', relative_path)

        checkpoint_row = {
            'file_id': file_id,
            'relative_path': relative_path,
            'session_id': session_id,
            'status': 'failed',
            'model_name': MODEL_NAME,
            'device': DEVICE,
            'compute_type': COMPUTE_TYPE,
            'language': None,
            'task': TASK,
            'runtime_seconds': float(runtime_seconds),
            'audio_duration_seconds': None,
            'raw_json_path': str(raw_json_path),
            'segments_json_path': str(segments_json_path),
            'transcript_path': str(transcript_path),
            'error': repr(exc),
            'updated_at': datetime.utcnow().isoformat(timespec='seconds') + 'Z',
        }
        checkpoint_df.loc[file_id] = checkpoint_row
        save_checkpoint(checkpoint_df.reset_index(drop=True), CHECKPOINT_INDEX)

        fail_count += 1

logging.info('ASR run complete: %s success, %s skipped, %s failed', success_count, skip_count, fail_count)
summary_df = pd.DataFrame({
    'metric': ['success', 'skipped', 'failed', 'total'],
    'value': [success_count, skip_count, fail_count, len(audio_files)],
})
summary_df

2026-05-31 16:55:28,903 [INFO] Loading checkpoint index: /Users/adityagoyal/SRH/Thesis Research/thesis_code/data/asr_outputs/checkpoints/asr_index.csv
2026-05-31 16:55:28,910 [INFO] Discovered 36 audio files
2026-05-31 16:55:28,911 [INFO] Initializing Whisper model: medium
2026-05-31 16:55:29,194 [INFO] HTTP Request: GET https://huggingface.co/api/models/Systran/faster-whisper-medium/revision/main "HTTP/1.1 200 OK"
2026-05-31 16:55:31,986 [INFO] [1/36] Transcribing: data/raw_audio/ES2014a.Mix-Headset.wav
2026-05-31 16:55:32,301 [INFO] Processing audio with duration 19:09.013
2026-05-31 16:55:37,850 [INFO] Detected language 'en' with probability 0.90
/var/folders/cg/1qj3kl_12w168jf9m109fgrr0000gn/T/ipykernel_49195/3912822684.py:69: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created_at': datetime.utcnow().isoformat(timespec='se

,metric,value
0,success,24
1,skipped,12
2,failed,0
3,total,36


## Output Structure
- raw/: per-file JSON with metadata, text, and segments
- segments/: per-file JSON with just segments and minimal metadata
- transcripts/: per-file transcript text
- checkpoints/asr_index.csv: master index and checkpoint table
- logs/: per-run log file